# Episode 4: Two Agents Working Together

One agent gives you answers. Two agents give you answers that have been questioned, challenged, and refined.

**In this episode, you'll build:** Two agents that collaborate — one researches, one reviews.

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from autogen import ConversableAgent, LLMConfig

## Why two agents?

Think about the last time you wrote something important — an email to a client, a design doc, a report. Did you catch all your own mistakes? Probably not. That's why we have editors, reviewers, and second opinions.

A single agent is limited by its system prompt. It plays one role and can only see things from one angle. Two agents with **different roles** can check each other's work — like a writer and an editor, or a researcher and a reviewer.

BetterFutureLabs found this approach can "shave days off the research and analysis process" through multi-agent discourse. The agents don't just divide work — they make each other better.

## Step 1: Create a researcher agent

Our researcher will gather facts on a topic. Notice how focused the system message is — one clear job.

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from autogen import ConversableAgent, LLMConfig

llm_config = LLMConfig({"model": "gpt-4o-mini"})

researcher = ConversableAgent(
    name="researcher",
    system_message=(
        "You are a researcher. When given a topic, provide 3 key facts about it. "
        "Be concise — one sentence per fact."
    ),
    llm_config=llm_config, human_input_mode="NEVER",
)

## Step 2: Create a reviewer agent

Now the counterpart — someone whose entire job is to poke holes. This separation of concerns is what makes the pattern powerful.

In [ ]:
reviewer = ConversableAgent(
    name="reviewer",
    system_message=(
        "You are a fact-checker. Review the researcher's facts. "
        "For each fact, say whether it seems accurate and why. "
        "End with an overall assessment."
    ),
    llm_config=llm_config,
        human_input_mode="NEVER",
)

## Step 3: Start the conversation

`run()` starts a conversation between the two agents. `max_turns` limits how many times they go back and forth — think of it as setting a timer for the meeting.

Let's see this in action:

In [ ]:
result = researcher.run(
    reviewer,
    message="Research the topic: Benefits of multi-agent AI systems",
    max_turns=2,
)
result.process()

print("--- Conversation Summary ---")
print(result.summary)

## What just happened?

Notice how the reviewer caught issues the researcher missed — that's the value of separation of concerns. Each agent stayed in its lane:

1. The **researcher** generated 3 key facts about multi-agent AI systems
2. The **reviewer** critiqued each fact for accuracy
3. They went back and forth for 3 turns, each improving the output

This is the simplest multi-agent pattern, and it's surprisingly effective. Two focused perspectives consistently beat one generalist trying to do everything.

## Key concepts

A few things to remember from this pattern:

- **`max_turns`** — controls how long the conversation runs (number of back-and-forth exchanges)
- **`result.process()`** — runs the conversation to completion (AG2 uses lazy evaluation)
- **`result.summary`** — gives a concise summary of the conversation (available after `.process()`)
- **`result.messages`** — contains the full conversation history as a list of messages

## Try it yourself

Here are some experiments to build your intuition:

1. **Change `max_turns` to 5** — does the quality improve with more iterations, or do the agents start going in circles?
2. **Change the researcher's system_message** to "always include one incorrect fact" — does the reviewer catch it?
3. **Try a different topic** — pick something you know well so you can judge the output yourself.

## Additional Content

### Termination conditions

Sometimes you want the agents to decide for themselves when the work is done, rather than relying on a fixed turn count. That's what `is_termination_msg` is for — the conversation stops when a specific condition is met.

```python
agent = ConversableAgent(
    name="reviewer",
    is_termination_msg=lambda msg: "APPROVED" in msg.get("content", ""),
    ...
)
```

Now the conversation stops as soon as the reviewer says "APPROVED" — a natural ending rather than an arbitrary cutoff.

### Summary methods

`result.summary` uses the last message by default. For longer conversations, you might want a proper summary instead. You can ask an LLM to read the full conversation and distill it:

```python
result = researcher.run(
    reviewer,
    message="Research the topic: Benefits of multi-agent AI systems",
    max_turns=3,
    summary_method="reflection_with_llm",
)
```

## What's Next

Two agents are powerful, but what happens when a problem needs three perspectives? Or four? How do you coordinate a whole team?

In the next episode, you'll expand to **3+ agents in a group chat** — and that's where orchestration really begins.

**Bonus:** Try the two-agent chat demo in the [AG2 Playground](https://playground.ag2.ai).